In [ ]:
from pathlib import Path
import os
import sys
import platform

import torch
import torchvision


# --------------------------------------------------
# 1. Detect the execution environment
# --------------------------------------------------

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


# --------------------------------------------------
# 2. Configure persistent storage
# --------------------------------------------------

if IN_COLAB:
    drive.mount("/content/drive")

    WORK_ROOT = Path(
        "/content/drive/MyDrive/DistillationResearch/Paper01_Imagenette"
    )

    # Temporary storage belonging to this Colab runtime.
    RUNTIME_DATA_ROOT = Path("/content/imagenette_runtime")

else:
    WORK_ROOT = Path(
        os.environ.get(
            "DISTILLATION_OUTPUT_DIR",
            "papers/hinton-2015-distillation/artifacts"
        )
    )

    RUNTIME_DATA_ROOT = WORK_ROOT / "runtime_data"


DATA_ARCHIVE_DIR = WORK_ROOT / "data"
CHECKPOINT_ROOT = WORK_ROOT / "checkpoints"
RESULTS_ROOT = WORK_ROOT / "results"
CONFIG_ROOT = WORK_ROOT / "config"


# Create the directories if they do not already exist.
for directory in [
    DATA_ARCHIVE_DIR,
    RUNTIME_DATA_ROOT,
    CHECKPOINT_ROOT,
    RESULTS_ROOT,
    CONFIG_ROOT,
]:
    directory.mkdir(parents=True, exist_ok=True)


# Dataset archive location.
DATASET_ARCHIVE_PATH = DATA_ARCHIVE_DIR / "imagenette2-160.tgz"


# --------------------------------------------------
# 3. Detect available compute
# --------------------------------------------------

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


# --------------------------------------------------
# 4. Display environment information
# --------------------------------------------------

print(f"Execution environment: {'Google Colab' if IN_COLAB else 'Local'}")
print(f"Python version: {sys.version.split()[0]}")
print(f"PyTorch version: {torch.__version__}")
print(f"TorchVision version: {torchvision.__version__}")
print(f"Device: {DEVICE}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"CUDA version: {torch.version.cuda}")
elif IN_COLAB:
    print("WARNING: No GPU is currently assigned to this Colab session.")

print(f"\nPersistent output directory: {WORK_ROOT}")
print(f"Runtime data directory: {RUNTIME_DATA_ROOT}")

In [ ]:

import shutil
import tarfile
import urllib.request
from pathlib import Path, PurePosixPath

# --------------------------------------------------
# 1. Dataset configuration
# --------------------------------------------------

DATASET_URL = (
    "https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-160.tgz"
)

ARCHIVE_ROOT_NAME = "imagenette2-160"

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

EXPECTED_COUNTS = {
    "train": 9469,
    "val": 3925,
}


# Persistent archive in Google Drive.
DATASET_ARCHIVE_PATH = (
    DATA_ARCHIVE_DIR / "imagenette2-160.tgz"
)

# Temporary copy on the Colab runtime.
LOCAL_ARCHIVE_PATH = (
    RUNTIME_DATA_ROOT / "imagenette2-160.tgz"
)

# Final extracted dataset location.
DATASET_ROOT = (
    RUNTIME_DATA_ROOT / ARCHIVE_ROOT_NAME
)

# Temporary extraction directory.
EXTRACTION_STAGING = (
    RUNTIME_DATA_ROOT / "_imagenette_extracting"
)

DATA_ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
RUNTIME_DATA_ROOT.mkdir(parents=True, exist_ok=True)


# --------------------------------------------------
# 2. Check whether an archive is a valid Imagenette
# --------------------------------------------------

def is_valid_imagenette_archive(archive_path):
    """Check that the archive contains Imagenette train/val folders."""

    if not archive_path.is_file():
        return False

    try:
        with tarfile.open(archive_path, "r:gz") as archive:
            members = archive.getmembers()

            has_train = False
            has_val = False

            for member in members:
                parts = PurePosixPath(member.name).parts

                if parts and parts[0] == ".":
                    parts = parts[1:]

                if not parts:
                    continue

                if (
                    len(parts) >= 2
                    and parts[0] == ARCHIVE_ROOT_NAME
                    and parts[1] == "train"
                ):
                    has_train = True

                if (
                    len(parts) >= 2
                    and parts[0] == ARCHIVE_ROOT_NAME
                    and parts[1] == "val"
                ):
                    has_val = True

            return has_train and has_val

    except (tarfile.TarError, OSError, EOFError):
        return False


# --------------------------------------------------
# 3. Download the archive if necessary
# --------------------------------------------------

def download_imagenette():
    """Download to a temporary file, validate, then replace the archive."""

    if is_valid_imagenette_archive(DATASET_ARCHIVE_PATH):
        print("Valid Imagenette archive already exists in Drive.")
        return

    # Remove any incomplete or invalid previous download.
    temporary_path = DATASET_ARCHIVE_PATH.with_suffix(".tgz.part")

    if temporary_path.exists():
        temporary_path.unlink()

    print("Downloading Imagenette v2 (160px)...")

    request = urllib.request.Request(
        DATASET_URL,
        headers={"User-Agent": "Mozilla/5.0"},
    )

    try:
        from tqdm.auto import tqdm

        show_progress = True
    except ImportError:
        show_progress = False

    try:
        with urllib.request.urlopen(
            request, timeout=60
        ) as response:

            total_bytes = int(
                response.headers.get("Content-Length", 0)
            )

            progress = (
                tqdm(
                    total=total_bytes or None,
                    unit="B",
                    unit_scale=True,
                    desc="Imagenette download",
                )
                if show_progress
                else None
            )

            downloaded_bytes = 0

            with open(temporary_path, "wb") as file:
                while True:
                    chunk = response.read(1024 * 1024)

                    if not chunk:
                        break

                    file.write(chunk)
                    downloaded_bytes += len(chunk)

                    if progress is not None:
                        progress.update(len(chunk))

            if progress is not None:
                progress.close()

        # Do not replace the persistent archive until validation succeeds.
        if not is_valid_imagenette_archive(temporary_path):
            raise RuntimeError(
                "Downloaded file is not a valid Imagenette archive."
            )

        temporary_path.replace(DATASET_ARCHIVE_PATH)

        print(
            f"Download complete: "
            f"{downloaded_bytes / (1024**2):.2f} MiB"
        )

    except Exception:
        if temporary_path.exists():
            temporary_path.unlink()
        raise


download_imagenette()


# --------------------------------------------------
# 4. Copy the archive to temporary runtime storage
# --------------------------------------------------

if is_valid_imagenette_archive(LOCAL_ARCHIVE_PATH):
    print("Valid local archive already exists in this runtime.")

else:
    print("Copying archive from Drive to runtime storage...")

    temporary_local_path = (
        RUNTIME_DATA_ROOT / "imagenette2-160.copying"
    )

    if temporary_local_path.exists():
        temporary_local_path.unlink()

    shutil.copy2(
        DATASET_ARCHIVE_PATH,
        temporary_local_path,
    )

    if not is_valid_imagenette_archive(temporary_local_path):
        temporary_local_path.unlink()

        raise RuntimeError(
            "The local archive copy failed validation."
        )

    temporary_local_path.replace(LOCAL_ARCHIVE_PATH)

    print("Local archive copy verified.")


# --------------------------------------------------
# 5. Validate an already extracted dataset
# --------------------------------------------------

def count_split_images(split_path):
    """Return the image count for each class directory."""

    if not split_path.is_dir():
        return {}

    counts = {}

    for class_directory in sorted(split_path.iterdir()):
        if not class_directory.is_dir():
            continue

        image_count = sum(
            1
            for file_path in class_directory.rglob("*")
            if file_path.is_file()
            and file_path.suffix.lower() in IMAGE_EXTENSIONS
        )

        counts[class_directory.name] = image_count

    return counts


def is_valid_extracted_dataset(dataset_root):
    """Check the extracted train/val directory structure."""

    if not dataset_root.is_dir():
        return False

    train_counts = count_split_images(
        dataset_root / "train"
    )

    val_counts = count_split_images(
        dataset_root / "val"
    )

    return (
        len(train_counts) == 10
        and len(val_counts) == 10
        and set(train_counts) == set(val_counts)
        and all(count > 0 for count in train_counts.values())
        and all(count > 0 for count in val_counts.values())
    )


# --------------------------------------------------
# 6. Extract the dataset only when necessary
# --------------------------------------------------

if is_valid_extracted_dataset(DATASET_ROOT):
    print("Valid extracted Imagenette dataset already exists.")

else:
    print("Extracting Imagenette...")

    if EXTRACTION_STAGING.exists():
        shutil.rmtree(EXTRACTION_STAGING)

    EXTRACTION_STAGING.mkdir(parents=True)

    try:
        with tarfile.open(
            LOCAL_ARCHIVE_PATH, "r:gz"
        ) as archive:

            members = archive.getmembers()

            # Basic archive safety checks:
            # reject path traversal and links.
            for member in members:
                member_path = PurePosixPath(member.name)

                if (
                    member_path.is_absolute()
                    or ".." in member_path.parts
                ):
                    raise ValueError(
                        f"Unsafe archive path: {member.name}"
                    )

                if member.issym() or member.islnk():
                    raise ValueError(
                        f"Unexpected archive link: {member.name}"
                    )

            archive.extractall(
                path=EXTRACTION_STAGING,
                members=members,
                filter="data",
            )

        staged_dataset = (
            EXTRACTION_STAGING / ARCHIVE_ROOT_NAME
        )

        if not is_valid_extracted_dataset(staged_dataset):
            raise RuntimeError(
                "Extracted dataset failed structural validation."
            )

        # Replace an invalid existing extraction only after
        # the new extraction has passed validation.
        if DATASET_ROOT.exists():
            shutil.rmtree(DATASET_ROOT)

        staged_dataset.replace(DATASET_ROOT)

        print("Extraction completed and verified.")

    finally:
        if EXTRACTION_STAGING.exists():
            shutil.rmtree(EXTRACTION_STAGING)


# --------------------------------------------------
# 7. Inspect the dataset
# --------------------------------------------------

train_counts = count_split_images(DATASET_ROOT / "train")
val_counts = count_split_images(DATASET_ROOT / "val")

print("\n" + "=" * 55)
print("IMAGENETTE DATASET SUMMARY")
print("=" * 55)

print(f"\nDataset location: {DATASET_ROOT}")
print(f"Number of classes: {len(train_counts)}")

print("\nClass-wise image counts:")

for class_name in sorted(train_counts):
    print(
        f"{class_name}: "
        f"train={train_counts[class_name]}, "
        f"val={val_counts[class_name]}"
    )

total_train = sum(train_counts.values())
total_val = sum(val_counts.values())

print("\nTotal images:")
print(f"Training:   {total_train:,}")
print(f"Validation: {total_val:,}")
print(f"Combined:   {total_train + total_val:,}")

# Compare observed counts with the documented v2 split.
if total_train != EXPECTED_COUNTS["train"]:
    print(
        "\nWARNING: Training count differs from the "
        "documented Imagenette v2 count."
    )

if total_val != EXPECTED_COUNTS["val"]:
    print(
        "\nWARNING: Validation count differs from the "
        "documented Imagenette v2 count."
    )

print("\nDataset preparation finished.")


In [ ]:

import random
import matplotlib.pyplot as plt

from PIL import Image


# --------------------------------------------------
# 1. Define the Imagenette class mapping
# --------------------------------------------------

CLASS_ID_TO_NAME = {
    "n01440764": "tench",
    "n02102040": "English springer",
    "n02979186": "cassette player",
    "n03000684": "chain saw",
    "n03028079": "church",
    "n03394916": "French horn",
    "n03417042": "garbage truck",
    "n03425413": "gas pump",
    "n03445777": "golf ball",
    "n03888257": "parachute",
}


# Keep class ordering deterministic.
CLASS_IDS = sorted(CLASS_ID_TO_NAME)

CLASS_TO_IDX = {
    class_id: index
    for index, class_id in enumerate(CLASS_IDS)
}

CLASS_NAMES = [
    CLASS_ID_TO_NAME[class_id]
    for class_id in CLASS_IDS
]


# Verify that the downloaded directories match our mapping.
assert set(CLASS_IDS) == set(train_counts.keys()), (
    "Class mapping does not match training directories."
)

assert set(CLASS_IDS) == set(val_counts.keys()), (
    "Class mapping does not match validation directories."
)


print("Class-to-index mapping:")

for class_id, index in CLASS_TO_IDX.items():
    print(
        f"{index}: {CLASS_ID_TO_NAME[class_id]} ({class_id})"
    )


# --------------------------------------------------
# 2. Select reproducible image samples
# --------------------------------------------------

rng = random.Random(42)

samples_per_class = 2

sample_records = []

for class_id in CLASS_IDS:

    class_directory = DATASET_ROOT / "train" / class_id

    image_paths = sorted(
        path
        for path in class_directory.rglob("*")
        if path.is_file()
        and path.suffix.lower() in IMAGE_EXTENSIONS
    )

    if len(image_paths) < samples_per_class:
        raise RuntimeError(
            f"Not enough images in class {class_id}."
        )

    selected_paths = rng.sample(
        image_paths,
        samples_per_class,
    )

    for image_path in selected_paths:

        # Open each image and convert it to RGB.
        with Image.open(image_path) as image:
            original_size = image.size
            image_rgb = image.convert("RGB")

        sample_records.append({
            "class_id": class_id,
            "class_name": CLASS_ID_TO_NAME[class_id],
            "path": image_path,
            "image": image_rgb,
            "width": original_size[0],
            "height": original_size[1],
        })


# --------------------------------------------------
# 3. Display the sample images
# --------------------------------------------------

fig, axes = plt.subplots(
    nrows=len(CLASS_IDS),
    ncols=samples_per_class,
    figsize=(10, 25),
)

for index, record in enumerate(sample_records):

    class_index = CLASS_IDS.index(record["class_id"])
    sample_index = index % samples_per_class

    ax = axes[class_index, sample_index]

    ax.imshow(record["image"])
    ax.set_title(
        f"{record['class_name']} — sample {sample_index + 1}"
    )
    ax.axis("off")


fig.suptitle(
    "Imagenette v2 — Training Image Samples",
    fontsize=16,
    y=1.002,
)

fig.tight_layout()

# Save the preview persistently to Google Drive.
PREVIEW_PATH = RESULTS_ROOT / "imagenette_dataset_preview.png"

fig.savefig(
    PREVIEW_PATH,
    dpi=150,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)


# --------------------------------------------------
# 4. Inspect original image dimensions
# --------------------------------------------------

print("\nOriginal sample image dimensions:")

short_sides = []
long_sides = []

for record in sample_records:

    width = record["width"]
    height = record["height"]

    short_sides.append(min(width, height))
    long_sides.append(max(width, height))

    print(
        f"{record['class_name']}: "
        f"{width} x {height}"
    )


print("\nDimension summary across sampled images:")
print(f"Shortest side: {min(short_sides)}–{max(short_sides)} px")
print(f"Longest side:  {min(long_sides)}–{max(long_sides)} px")

print(f"\nPreview saved to: {PREVIEW_PATH}")


In [ ]:

import json
import random
import numpy as np
import matplotlib.pyplot as plt

from collections import defaultdict
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset


# ==================================================
# 1. Reproducibility and configuration
# ==================================================

SEED = 42

INPUT_SIZE = 160
BATCH_SIZE = 32
INTERNAL_VAL_FRACTION = 0.10

# Use 2 workers in Colab and a simpler setting locally.
NUM_WORKERS = 2 if IN_COLAB else 0

IMAGE_MEAN = (0.485, 0.456, 0.406)
IMAGE_STD = (0.229, 0.224, 0.225)

TRAIN_CROP_SCALE = (0.75, 1.0)
TRAIN_CROP_RATIO = (0.75, 1.333)
HORIZONTAL_FLIP_PROBABILITY = 0.5

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

PIN_MEMORY = DEVICE.type == "cuda"


# ==================================================
# 2. Define image transformations
# ==================================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(
        size=INPUT_SIZE,
        scale=TRAIN_CROP_SCALE,
        ratio=TRAIN_CROP_RATIO,
        antialias=True,
    ),

    transforms.RandomHorizontalFlip(
        p=HORIZONTAL_FLIP_PROBABILITY
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD,
    ),
])


eval_transform = transforms.Compose([
    transforms.Resize(
        INPUT_SIZE,
        antialias=True,
    ),

    transforms.CenterCrop(INPUT_SIZE),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=IMAGE_MEAN,
        std=IMAGE_STD,
    ),
])


# ==================================================
# 3. Load the three dataset views
# ==================================================

TRAIN_DIRECTORY = DATASET_ROOT / "train"

# The source folder is named "val".
# In this experiment, we reserve it as the final test set.
OFFICIAL_VAL_DIRECTORY = DATASET_ROOT / "val"


# Training images with random augmentation.
train_dataset = datasets.ImageFolder(
    root=TRAIN_DIRECTORY,
    transform=train_transform,
)


# Same training images with deterministic evaluation transforms.
internal_val_dataset = datasets.ImageFolder(
    root=TRAIN_DIRECTORY,
    transform=eval_transform,
)


# Official source validation split, reserved for final evaluation.
test_dataset = datasets.ImageFolder(
    root=OFFICIAL_VAL_DIRECTORY,
    transform=eval_transform,
)


# Verify consistent class ordering.
assert train_dataset.class_to_idx == CLASS_TO_IDX
assert internal_val_dataset.class_to_idx == CLASS_TO_IDX
assert test_dataset.class_to_idx == CLASS_TO_IDX

assert train_dataset.targets == internal_val_dataset.targets

print("All three dataset views have a consistent class mapping.")


# ==================================================
# 4. Create a deterministic stratified split
# ==================================================

indices_by_class = defaultdict(list)

for image_index, class_index in enumerate(train_dataset.targets):
    indices_by_class[class_index].append(image_index)


split_rng = random.Random(SEED)

train_indices = []
internal_val_indices = []

train_indices_by_class = defaultdict(list)
internal_val_indices_by_class = defaultdict(list)


for class_index in range(len(CLASS_NAMES)):

    class_indices = indices_by_class[class_index].copy()

    split_rng.shuffle(class_indices)

    validation_count = round(
        len(class_indices) * INTERNAL_VAL_FRACTION
    )

    class_val_indices = class_indices[:validation_count]
    class_train_indices = class_indices[validation_count:]

    internal_val_indices.extend(class_val_indices)
    train_indices.extend(class_train_indices)

    internal_val_indices_by_class[class_index].extend(
        class_val_indices
    )

    train_indices_by_class[class_index].extend(
        class_train_indices
    )


# Keep the subset order deterministic.
train_indices.sort()
internal_val_indices.sort()


# Create the actual subsets.
train_subset = Subset(
    train_dataset,
    train_indices,
)

internal_val_subset = Subset(
    internal_val_dataset,
    internal_val_indices,
)


# Look up an original dataset index's position within each subset.
train_position_map = {
    original_index: subset_position
    for subset_position, original_index in enumerate(train_indices)
}

val_position_map = {
    original_index: subset_position
    for subset_position, original_index in enumerate(internal_val_indices)
}


# ==================================================
# 5. Create DataLoaders
# ==================================================

def seed_data_worker(worker_id):
    """Give each DataLoader worker reproducible Python/NumPy seeds."""

    worker_seed = torch.initial_seed() % (2**32)

    np.random.seed(worker_seed)
    random.seed(worker_seed)


# The training generator is reserved for actual training.
# Its state will later be included in resumable checkpoints.
TRAIN_LOADER_GENERATOR = torch.Generator()
TRAIN_LOADER_GENERATOR.manual_seed(SEED)


INTERNAL_VAL_GENERATOR = torch.Generator()
INTERNAL_VAL_GENERATOR.manual_seed(SEED + 1)


TEST_GENERATOR = torch.Generator()
TEST_GENERATOR.manual_seed(SEED + 2)


train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
    worker_init_fn=seed_data_worker,
    generator=TRAIN_LOADER_GENERATOR,
    persistent_workers=False,
)


internal_val_loader = DataLoader(
    internal_val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
    worker_init_fn=seed_data_worker,
    generator=INTERNAL_VAL_GENERATOR,
    persistent_workers=False,
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=PIN_MEMORY,
    drop_last=False,
    worker_init_fn=seed_data_worker,
    generator=TEST_GENERATOR,
    persistent_workers=False,
)


# ==================================================
# 6. Report split statistics
# ==================================================

train_split_counts = {
    CLASS_NAMES[index]: len(train_indices_by_class[index])
    for index in range(len(CLASS_NAMES))
}

internal_val_counts = {
    CLASS_NAMES[index]: len(internal_val_indices_by_class[index])
    for index in range(len(CLASS_NAMES))
}


print("\nDATASET SPLIT SUMMARY")
print("=" * 60)

for class_name in CLASS_NAMES:
    print(
        f"{class_name:20s} "
        f"train={train_split_counts[class_name]:4d} "
        f"internal_val={internal_val_counts[class_name]:4d}"
    )


print("\nTotal images:")
print(f"Training:            {len(train_subset):,}")
print(f"Internal validation: {len(internal_val_subset):,}")
print(f"Held-out test:       {len(test_dataset):,}")

assert len(train_subset) + len(internal_val_subset) == len(train_dataset)
assert len(test_dataset) == 3925


# ==================================================
# 7. Save experiment data configuration
# ==================================================

data_config = {
    "dataset": "Imagenette v2 160px",
    "dataset_url": DATASET_URL,
    "seed": SEED,
    "input_size": INPUT_SIZE,
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "internal_validation_fraction": INTERNAL_VAL_FRACTION,

    "class_to_index": CLASS_TO_IDX,
    "class_names": CLASS_NAMES,

    "normalization": {
        "mean": IMAGE_MEAN,
        "std": IMAGE_STD,
    },

    "training_transforms": [
        "RandomResizedCrop",
        "RandomHorizontalFlip",
        "ToTensor",
        "Normalize",
    ],

    "evaluation_transforms": [
        "Resize",
        "CenterCrop",
        "ToTensor",
        "Normalize",
    ],

    # Record the actual parameters, not only transform names.
    "transform_parameters": {
        "training": {
            "random_resized_crop": {
                "size": INPUT_SIZE,
                "scale": list(TRAIN_CROP_SCALE),
                "ratio": list(TRAIN_CROP_RATIO),
                "antialias": True,
            },
            "random_horizontal_flip": {
                "p": HORIZONTAL_FLIP_PROBABILITY,
            },
            "normalization": {
                "mean": list(IMAGE_MEAN),
                "std": list(IMAGE_STD),
            },
        },
        "evaluation": {
            "resize": {
                "size": INPUT_SIZE,
                "antialias": True,
            },
            "center_crop": {
                "size": INPUT_SIZE,
            },
            "normalization": {
                "mean": list(IMAGE_MEAN),
                "std": list(IMAGE_STD),
            },
        },
    },

    "split_definition": {
        "train": "90% stratified subset of original train folder",
        "internal_validation": "10% stratified subset of original train folder",
        "test": "Official Imagenette source val folder, held out for final evaluation",
    },

    "split_sizes": {
        "train": len(train_subset),
        "internal_validation": len(internal_val_subset),
        "test": len(test_dataset),
    },

    "per_class_counts": {
        "train": train_split_counts,
        "internal_validation": internal_val_counts,
    },
}


DATA_CONFIG_PATH = CONFIG_ROOT / "imagenette_data_config.json"

with open(DATA_CONFIG_PATH, "w") as file:
    json.dump(data_config, file, indent=4)


print(f"\nData configuration saved to: {DATA_CONFIG_PATH}")


# ==================================================
# 8. Visualise transformed samples
#    (training vs internal validation)
# ==================================================

def denormalize_for_display(image_tensor):

    mean = torch.tensor(IMAGE_MEAN).view(3, 1, 1)
    std = torch.tensor(IMAGE_STD).view(3, 1, 1)

    image_tensor = image_tensor.clone()

    image_tensor = image_tensor * std + mean

    return (
        image_tensor
        .permute(1, 2, 0)
        .clamp(0, 1)
        .numpy()
    )


# Preserve RNG states so visual inspection does not
# influence the subsequent model initialisation.
cpu_rng_state = torch.get_rng_state()

cuda_rng_state = (
    torch.cuda.get_rng_state_all()
    if torch.cuda.is_available()
    else None
)

python_rng_state = random.getstate()
numpy_rng_state = np.random.get_state()

try:

    torch.manual_seed(SEED + 200)

    fig, axes = plt.subplots(
        nrows=2,
        ncols=len(CLASS_NAMES),
        figsize=(24, 8),
        squeeze=False,
    )

    for class_index, class_name in enumerate(CLASS_NAMES):

        # One training example from this class.
        original_train_index = train_indices_by_class[class_index][0]

        train_position = train_position_map[original_train_index]

        train_image, train_label = train_subset[train_position]


        # One internal-validation example from this class.
        original_val_index = internal_val_indices_by_class[class_index][0]

        val_position = val_position_map[original_val_index]

        val_image, val_label = internal_val_subset[val_position]


        # Display training transform.
        axes[0, class_index].imshow(
            denormalize_for_display(train_image)
        )

        axes[0, class_index].set_title(
            f"Train: {class_name}"
        )

        axes[0, class_index].axis("off")


        # Display deterministic validation transform.
        axes[1, class_index].imshow(
            denormalize_for_display(val_image)
        )

        axes[1, class_index].set_title(
            f"Validation: {class_name}"
        )

        axes[1, class_index].axis("off")


    fig.suptitle(
        "Imagenette — Training and Validation Transforms",
        fontsize=15,
    )

    fig.tight_layout()

    TRANSFORM_PREVIEW_PATH = (
        RESULTS_ROOT / "imagenette_transform_preview.png"
    )

    fig.savefig(
        TRANSFORM_PREVIEW_PATH,
        dpi=150,
        bbox_inches="tight",
    )

    plt.show()
    plt.close(fig)

finally:

    torch.set_rng_state(cpu_rng_state)

    if cuda_rng_state is not None:
        torch.cuda.set_rng_state_all(cuda_rng_state)

    random.setstate(python_rng_state)
    np.random.set_state(numpy_rng_state)


print(f"\nTransformed preview saved to: {TRANSFORM_PREVIEW_PATH}")
print("\nData pipeline setup completed.")


In [ ]:

import copy


# ==================================================
# 1. Model configuration
# ==================================================

NUM_CLASSES = len(CLASS_NAMES)
MODEL_INIT_SEED = SEED

# All models will run on the previously detected device.
# No pretrained weights will be downloaded or loaded.


# ==================================================
# 2. Initialise the teacher and hard-label student
# ==================================================

# Preserve the existing random-number generator states.
# Model initialisation should be reproducible without
# unexpectedly changing the RNG state used elsewhere.

with torch.random.fork_rng(
    devices=list(range(torch.cuda.device_count()))
):

    torch.manual_seed(MODEL_INIT_SEED)

    # ResNet-34 teacher.
    teacher_model = torchvision.models.resnet34(
        weights=None,
        num_classes=NUM_CLASSES,
    )

    # ResNet-18 hard-label baseline student.
    student_hard_model = torchvision.models.resnet18(
        weights=None,
        num_classes=NUM_CLASSES,
    )


# ==================================================
# 3. Create the two distilled students
#    from the exact same initial student state
# ==================================================

# Deepcopy creates independent model objects with
# identical parameters and BatchNorm buffers.

student_kd_soft_model = copy.deepcopy(
    student_hard_model
)

student_kd_combined_model = copy.deepcopy(
    student_hard_model
)


# Keep an independent snapshot of the shared initial state.
# We can use this later to verify or restore initialisation.

STUDENT_INITIAL_STATE = copy.deepcopy(
    student_hard_model.state_dict()
)


# ==================================================
# 4. Move all models to the selected device
# ==================================================

MODEL_REGISTRY = {
    "teacher": teacher_model,
    "student_hard": student_hard_model,
    "student_kd_soft": student_kd_soft_model,
    "student_kd_combined": student_kd_combined_model,
}

for model in MODEL_REGISTRY.values():
    model.to(DEVICE)


# ==================================================
# 5. Verify classifier heads
# ==================================================

for model_name, model in MODEL_REGISTRY.items():

    assert model.fc.in_features == 512
    assert model.fc.out_features == NUM_CLASSES

print("All classifier heads have been configured correctly.")


# ==================================================
# 6. Verify identical student initialisation
# ==================================================

reference_state = student_hard_model.state_dict()

for student_name, student_model in {
    "student_kd_soft": student_kd_soft_model,
    "student_kd_combined": student_kd_combined_model,
}.items():

    candidate_state = student_model.state_dict()

    assert reference_state.keys() == candidate_state.keys()

    for parameter_name in reference_state:

        reference_tensor = reference_state[parameter_name]
        candidate_tensor = candidate_state[parameter_name]

        # Values must be identical.
        assert torch.equal(
            reference_tensor,
            candidate_tensor,
        ), f"Initialisation mismatch: {student_name}, {parameter_name}"

        # Storage must be independent.
        assert reference_tensor.data_ptr() != candidate_tensor.data_ptr(), (
            f"Shared storage detected: {student_name}, {parameter_name}"
        )


print("Both distilled students have identical initial states.")
print("Student model parameters and buffers have independent storage.")


# ==================================================
# 7. Calculate and report parameter counts
# ==================================================

def count_parameters(model):
    """Count all trainable model parameters."""

    return sum(
        parameter.numel()
        for parameter in model.parameters()
        if parameter.requires_grad
    )


print("\nMODEL PARAMETER SUMMARY")
print("=" * 65)

for model_name, model in MODEL_REGISTRY.items():

    parameter_count = count_parameters(model)

    print(
        f"{model_name:24s} "
        f"{parameter_count:>15,} parameters"
    )


# ==================================================
# 8. Verify forward-pass output dimensions
# ==================================================

# Use a small dummy batch for an architecture smoke test.
# No actual dataset images or training updates are involved.

dummy_batch = torch.zeros(
    2,
    3,
    INPUT_SIZE,
    INPUT_SIZE,
    device=DEVICE,
)


print("\nMODEL FORWARD-PASS CHECK")
print("=" * 65)

for model_name, model in MODEL_REGISTRY.items():

    previous_mode = model.training

    try:

        model.eval()

        with torch.inference_mode():

            logits = model(dummy_batch)

        assert logits.shape == (2, NUM_CLASSES)

        print(
            f"{model_name:24s} "
            f"output shape: {tuple(logits.shape)}"
        )

    finally:

        model.train(previous_mode)


del dummy_batch

print("\nModel architecture setup completed.")


In [ ]:

# ==================================================
# Cell 6 — Experiment configuration
# ==================================================

# 1. Shared training configuration

NUM_EPOCHS = 90

OPTIMIZER_NAME = "SGD"

LEARNING_RATE = 0.0125
MOMENTUM = 0.9
WEIGHT_DECAY = 1e-4

SCHEDULER_NAME = "StepLR"
SCHEDULER_STEP_SIZE = 30
SCHEDULER_GAMMA = 0.1


# 2. Knowledge distillation configuration

KD_TEMPERATURE = 4.0

# This coefficient represents the hard-label loss weight.
HARD_LABEL_WEIGHT = 0.1
SOFT_TARGET_WEIGHT = 1.0 - HARD_LABEL_WEIGHT


# 3. Define the objective for each training stage

STAGE_OBJECTIVES = {
    "teacher": "hard_label_cross_entropy",
    "student_hard": "hard_label_cross_entropy",
    "student_kd_soft": "soft_target_kl_divergence",
    "student_kd_combined": "combined_kd_and_cross_entropy",
}


# 4. Create persistent checkpoint directories

CHECKPOINT_ROOT = CONFIG_ROOT.parent / "checkpoints"

CHECKPOINT_DIRS = {
    stage_name: CHECKPOINT_ROOT / stage_name
    for stage_name in MODEL_REGISTRY
}

for checkpoint_directory in CHECKPOINT_DIRS.values():
    checkpoint_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# 5. Construct the experiment configuration

EXPERIMENT_CONFIG = {
    "experiment": "Hinton Knowledge Distillation on Imagenette",
    "paper": "Distilling the Knowledge in a Neural Network (2015)",

    "dataset_config": str(DATA_CONFIG_PATH),

    "device": str(DEVICE),
    "seed": SEED,
    "model_initialisation_seed": MODEL_INIT_SEED,

    "models": {
        name: {
            "architecture": (
                "ResNet-34" if name == "teacher" else "ResNet-18"
            ),
            "num_classes": NUM_CLASSES,
            "pretrained_weights": None,
            "trainable_parameters": count_parameters(model),
        }
        for name, model in MODEL_REGISTRY.items()
    },

    "training": {
        "batch_size": BATCH_SIZE,
        "epochs": NUM_EPOCHS,
        "optimizer": OPTIMIZER_NAME,
        "learning_rate": LEARNING_RATE,
        "momentum": MOMENTUM,
        "weight_decay": WEIGHT_DECAY,
        "scheduler": SCHEDULER_NAME,
        "scheduler_step_size": SCHEDULER_STEP_SIZE,
        "scheduler_gamma": SCHEDULER_GAMMA,
    },

    "distillation": {
        "temperature": KD_TEMPERATURE,
        "hard_label_weight": HARD_LABEL_WEIGHT,
        "soft_target_weight": SOFT_TARGET_WEIGHT,
        "soft_only_uses_hard_labels": False,
    },

    "stage_objectives": STAGE_OBJECTIVES,

    "checkpointing": {
        "save_after_each_completed_epoch": True,
        "latest_checkpoint": "latest.pt",
        "best_model_checkpoint": "best_model.pt",
        "resume_uses_latest_checkpoint": True,
    },

    "checkpoint_directories": {
        name: str(path)
        for name, path in CHECKPOINT_DIRS.items()
    },
}


# 6. Save the experiment configuration

EXPERIMENT_CONFIG_PATH = (
    CONFIG_ROOT / "paper01_experiment_config.json"
)

with open(EXPERIMENT_CONFIG_PATH, "w") as file:
    json.dump(
        EXPERIMENT_CONFIG,
        file,
        indent=4,
    )


# 7. Verify configuration

assert len(MODEL_REGISTRY) == 4
assert BATCH_SIZE == 32
assert SOFT_TARGET_WEIGHT + HARD_LABEL_WEIGHT == 1.0

assert all(
    path.exists()
    for path in CHECKPOINT_DIRS.values()
)

print("EXPERIMENT CONFIGURATION")
print("=" * 60)

print(f"Models:              {len(MODEL_REGISTRY)}")
print(f"Epochs per model:    {NUM_EPOCHS}")
print(f"Optimizer:           {OPTIMIZER_NAME}")
print(f"Learning rate:       {LEARNING_RATE}")
print(f"Batch size:          {BATCH_SIZE}")
print(f"KD temperature:      {KD_TEMPERATURE}")
print(f"Hard-label weight:   {HARD_LABEL_WEIGHT}")
print(f"Soft-target weight:  {SOFT_TARGET_WEIGHT}")

print(f"\nCheckpoint root: {CHECKPOINT_ROOT}")

print(f"\nConfiguration saved to: {EXPERIMENT_CONFIG_PATH}")
print("\nExperiment configuration completed.")


In [ ]:

import torch.nn.functional as F


# ==================================================
# 1. Hard-label cross-entropy loss
# ==================================================

def hard_label_loss(student_logits, labels):
    """
    Calculate ordinary cross-entropy loss.

    Used for:
    - Teacher training
    - Hard-label student training
    """

    if student_logits.ndim != 2:
        raise ValueError("Logits must have shape (batch_size, num_classes).")

    if labels.ndim != 1:
        raise ValueError("Labels must have shape (batch_size,).")

    if student_logits.shape[0] != labels.shape[0]:
        raise ValueError("Logits and labels must have matching batch sizes.")

    return F.cross_entropy(student_logits, labels)


# ==================================================
# 2. Soft-target knowledge distillation loss
# ==================================================

def soft_target_kd_loss(
    student_logits,
    teacher_logits,
    temperature=KD_TEMPERATURE,
):
    """
    Calculate temperature-scaled KL divergence
    between teacher and student probability distributions.
    """

    if student_logits.ndim != 2 or teacher_logits.ndim != 2:
        raise ValueError("Teacher and student logits must be 2D.")

    if student_logits.shape != teacher_logits.shape:
        raise ValueError("Teacher and student logits must have matching shapes.")

    if temperature <= 0:
        raise ValueError("Temperature must be greater than zero.")

    # Teacher probabilities at temperature T.
    # Detach ensures that no gradients flow into the teacher.
    teacher_probabilities = F.softmax(
        teacher_logits.detach() / temperature,
        dim=1,
    )

    # Student log-probabilities at temperature T.
    student_log_probabilities = F.log_softmax(
        student_logits / temperature,
        dim=1,
    )

    # KL divergence: teacher distribution || student distribution.
    kd_loss = F.kl_div(
        student_log_probabilities,
        teacher_probabilities,
        reduction="batchmean",
    )

    # Temperature-squared correction from Hinton-style KD.
    return (temperature ** 2) * kd_loss


# ==================================================
# 3. Combined KD and hard-label loss
# ==================================================

def combined_kd_loss(
    student_logits,
    teacher_logits,
    labels,
):
    """
    Combine soft-target distillation with
    ordinary ground-truth supervision.

    Uses weights from EXPERIMENT_CONFIG.
    """

    kd_component = soft_target_kd_loss(
        student_logits,
        teacher_logits,
    )

    hard_component = hard_label_loss(
        student_logits,
        labels,
    )

    total_loss = (
        SOFT_TARGET_WEIGHT * kd_component
        + HARD_LABEL_WEIGHT * hard_component
    )

    return total_loss, kd_component, hard_component


print("All three loss functions have been defined.")


In [ ]:

import os
import warnings
from pathlib import Path


# ==================================================
# 1. Capture all relevant RNG states
# ==================================================

def capture_rng_state():
    """
    Capture Python, NumPy, CPU Torch and CUDA RNG states.

    NumPy's state is converted into ordinary Python
    values so that it can be loaded with weights_only=True.
    """

    numpy_state = np.random.get_state()

    rng_state = {
        "python": random.getstate(),

        "numpy": {
            "algorithm": numpy_state[0],
            "keys": numpy_state[1].tolist(),
            "position": int(numpy_state[2]),
            "has_gauss": int(numpy_state[3]),
            "cached_gaussian": float(numpy_state[4]),
        },

        "torch_cpu": torch.get_rng_state().clone(),

        "torch_cuda": (
            [
                state.clone()
                for state in torch.cuda.get_rng_state_all()
            ]
            if torch.cuda.is_available()
            else []
        ),
    }

    return rng_state


# ==================================================
# 2. Restore all relevant RNG states
# ==================================================

def restore_rng_state(rng_state):
    """
    Restore Python, NumPy, CPU Torch and CUDA RNG states.
    """

    random.setstate(rng_state["python"])

    numpy_state = rng_state["numpy"]

    np.random.set_state((
        numpy_state["algorithm"],
        np.asarray(
            numpy_state["keys"],
            dtype=np.uint32,
        ),
        numpy_state["position"],
        numpy_state["has_gauss"],
        numpy_state["cached_gaussian"],
    ))

    torch.set_rng_state(
        rng_state["torch_cpu"].cpu()
    )

    saved_cuda_states = rng_state["torch_cuda"]

    if saved_cuda_states:

        if not torch.cuda.is_available():

            warnings.warn(
                "Checkpoint contains CUDA RNG states, "
                "but CUDA is unavailable. CUDA randomness "
                "cannot be restored.",
                RuntimeWarning,
            )

        elif len(saved_cuda_states) != torch.cuda.device_count():

            warnings.warn(
                "Saved CUDA device count differs from the "
                "current runtime. Exact CUDA RNG restoration "
                "is not guaranteed.",
                RuntimeWarning,
            )

        else:

            torch.cuda.set_rng_state_all(
                saved_cuda_states
            )


# ==================================================
# 3. Atomic checkpoint saving
# ==================================================

def atomic_torch_save(payload, destination):
    """
    Save a checkpoint to a temporary file in the same
    directory, then replace the destination.

    An incomplete temporary write will not replace
    the previously saved checkpoint.
    """

    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    temporary_path = destination.with_name(
        f".{destination.name}.tmp"
    )

    try:

        torch.save(
            payload,
            temporary_path,
        )

        os.replace(
            temporary_path,
            destination,
        )

    finally:

        if temporary_path.exists():
            temporary_path.unlink()

    return destination

print("RNG and checkpoint utilities defined.")


In [ ]:

# ==================================================
# Cell 10 — DataLoader RNG and experiment compatibility
# ==================================================

# 1. Register the generators belonging to our actual DataLoaders.

DATA_LOADER_GENERATORS = {
    "train": train_loader.generator,
    "internal_validation": internal_val_loader.generator,
    "test": test_loader.generator,
}

assert all(
    generator is not None
    for generator in DATA_LOADER_GENERATORS.values()
)


# 2. Capture the original state of every generator.
#
# These are the starting states for a FRESH training stage.
# We capture them before any actual training begins.

INITIAL_DATALOADER_GENERATOR_STATES = {
    name: generator.get_state().clone()
    for name, generator in DATA_LOADER_GENERATORS.items()
}


# 3. Capture the current generator states.
#
# These are the states that will go into latest.pt.

def capture_dataloader_generator_states():

    return {
        name: generator.get_state().clone()
        for name, generator in DATA_LOADER_GENERATORS.items()
    }


# 4. Restore generator states from a checkpoint.

def restore_dataloader_generator_states(generator_states):

    expected_names = set(DATA_LOADER_GENERATORS)
    saved_names = set(generator_states)

    if expected_names != saved_names:
        raise ValueError(
            f"DataLoader generator mismatch. "
            f"Expected {expected_names}, received {saved_names}."
        )

    for name, generator in DATA_LOADER_GENERATORS.items():

        generator.set_state(
            generator_states[name].cpu()
        )


# 5. Reset all DataLoaders before a NEW experiment stage.
#
# IMPORTANT:
# Use this for a fresh stage, NOT when resuming a checkpoint.

def reset_dataloader_generators_for_new_stage():

    restore_dataloader_generator_states(
        INITIAL_DATALOADER_GENERATOR_STATES
    )


# 6. Construct a stage-specific compatibility configuration.
#
# This records the settings that must remain compatible
# when restoring a checkpoint.

def build_stage_compatibility_config(stage_name):

    if stage_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown training stage: {stage_name}")

    architecture = (
        "ResNet-34" if stage_name == "teacher" else "ResNet-18"
    )

    config = {
        "stage_name": stage_name,

        "architecture": architecture,

        "num_classes": NUM_CLASSES,

        "class_to_idx": copy.deepcopy(CLASS_TO_IDX),

        "dataset": {
            "name": data_config["dataset"],
            "seed": data_config["seed"],
            "input_size": data_config["input_size"],
            "split_sizes": copy.deepcopy(
                data_config["split_sizes"]
            ),
            "transform_parameters": copy.deepcopy(
                data_config["transform_parameters"]
            ),
        },

        "training": copy.deepcopy(
            EXPERIMENT_CONFIG["training"]
        ),

        "objective": STAGE_OBJECTIVES[stage_name],
    }

    # Only KD stages depend on temperature and loss weights.
    if stage_name in (
        "student_kd_soft",
        "student_kd_combined",
    ):

        config["distillation"] = {
            "temperature": KD_TEMPERATURE,
        }

    if stage_name == "student_kd_combined":

        config["distillation"].update({
            "hard_label_weight": HARD_LABEL_WEIGHT,
            "soft_target_weight": SOFT_TARGET_WEIGHT,
        })

    return config


# 7. Prepare compatibility configurations for all stages.

STAGE_COMPATIBILITY_CONFIGS = {
    stage_name: build_stage_compatibility_config(stage_name)
    for stage_name in MODEL_REGISTRY
}


# ==================================================
# 8. Verify DataLoader generator reproducibility
# ==================================================

original_generator_states = capture_dataloader_generator_states()

try:

    # Generate reference sequences from each generator.
    expected_sequences = {
        name: torch.rand(5, generator=generator)
        for name, generator in DATA_LOADER_GENERATORS.items()
    }

    # Restore the original states.
    restore_dataloader_generator_states(
        original_generator_states
    )

    # Generate again. The sequences must match.
    actual_sequences = {
        name: torch.rand(5, generator=generator)
        for name, generator in DATA_LOADER_GENERATORS.items()
    }

    for name in DATA_LOADER_GENERATORS:

        assert torch.equal(
            expected_sequences[name],
            actual_sequences[name],
        ), f"Generator reproducibility failed: {name}"

    print("DataLoader generator verification: Passed")

finally:

    # Do not leave the actual experiment generators advanced.
    restore_dataloader_generator_states(
        original_generator_states
    )


# ==================================================
# 9. Verify fresh-stage reset
# ==================================================

reset_dataloader_generators_for_new_stage()

for name, generator in DATA_LOADER_GENERATORS.items():

    assert torch.equal(
        generator.get_state(),
        INITIAL_DATALOADER_GENERATOR_STATES[name],
    )

# Restore the state that existed before this test.
restore_dataloader_generator_states(
    original_generator_states
)


# ==================================================
# 10. Verify stage-specific configurations
# ==================================================

assert len(STAGE_COMPATIBILITY_CONFIGS) == 4

assert (
    STAGE_COMPATIBILITY_CONFIGS["teacher"]["architecture"]
    == "ResNet-34"
)

assert (
    STAGE_COMPATIBILITY_CONFIGS["student_hard"]["architecture"]
    == "ResNet-18"
)

assert (
    STAGE_COMPATIBILITY_CONFIGS["student_kd_soft"]["objective"]
    == "soft_target_kl_divergence"
)

assert (
    STAGE_COMPATIBILITY_CONFIGS["student_kd_combined"]["objective"]
    == "combined_kd_and_cross_entropy"
)


print("\nEXPERIMENT COMPATIBILITY VERIFICATION")
print("=" * 60)

for stage_name, config in STAGE_COMPATIBILITY_CONFIGS.items():

    print(
        f"{stage_name:24s} "
        f"{config['architecture']:12s} "
        f"{config['objective']}"
    )

print("\nFresh-stage generator reset: Passed")
print("Stage compatibility configuration: Passed")

print("\nDataLoader and compatibility setup completed.")


In [ ]:


CHECKPOINT_VERSION = 1


# ==================================================
# 1. Create a portable model state dictionary
# ==================================================

def get_cpu_model_state(model):
    """
    Create an independent CPU copy of model parameters
    and persistent buffers.
    """

    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in model.state_dict().items()
    }


# ==================================================
# 2. Build a complete training checkpoint
# ==================================================

def build_training_checkpoint(
    stage_name,
    model,
    optimizer,
    epoch_completed,
    best_metric=None,
    best_epoch=None,
    scheduler=None,
    scaler=None,
    history=None,
    epoch_metrics=None,
):
    """
    Construct the complete state required to resume
    training at the next epoch.
    """

    if stage_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown stage: {stage_name}")

    if epoch_completed < 1 or epoch_completed > NUM_EPOCHS:
        raise ValueError("Invalid completed epoch.")

    if best_epoch is not None:
        if best_epoch < 1 or best_epoch > epoch_completed:
            raise ValueError("Invalid best epoch.")

    if best_metric is None and best_epoch is not None:
        raise ValueError("Best epoch requires a best metric.")

    checkpoint = {

        "checkpoint_version": CHECKPOINT_VERSION,

        "checkpoint_type": "latest_training_state",

        "stage_name": stage_name,

        # Model architecture and learned state.
        "model_state_dict": get_cpu_model_state(model),

        # Optimisation state.
        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": (
            scheduler.state_dict()
            if scheduler is not None
            else None
        ),

        "scaler_state_dict": (
            scaler.state_dict()
            if scaler is not None
            else None
        ),

        # Epoch progress.
        "epoch_completed": int(epoch_completed),

        "next_epoch": int(epoch_completed + 1),

        "total_epochs": NUM_EPOCHS,

        # Best validation result so far.
        "best_metric": (
            float(best_metric)
            if best_metric is not None
            else None
        ),

        "best_epoch": best_epoch,

        # Experiment records.
        "training_history": copy.deepcopy(
            history if history is not None else []
        ),

        "epoch_metrics": copy.deepcopy(
            epoch_metrics if epoch_metrics is not None else {}
        ),

        # Reproducibility state.
        "rng_state": capture_rng_state(),

        "dataloader_generator_states": (
            capture_dataloader_generator_states()
        ),

        # Configuration and compatibility.
        "experiment_config": copy.deepcopy(
            EXPERIMENT_CONFIG
        ),

        "stage_compatibility_config": copy.deepcopy(
            STAGE_COMPATIBILITY_CONFIGS[stage_name]
        ),

        "class_to_idx": copy.deepcopy(CLASS_TO_IDX),
    }

    return checkpoint


# ==================================================
# 3. Save latest training checkpoint
# ==================================================

def save_latest_checkpoint(
    stage_name,
    model,
    optimizer,
    epoch_completed,
    **kwargs,
):
    """
    Save the latest completed training epoch.

    This is the checkpoint used for resuming training.
    """

    if stage_name not in CHECKPOINT_DIRS:
        raise ValueError(f"Unknown stage: {stage_name}")

    checkpoint = build_training_checkpoint(
        stage_name=stage_name,
        model=model,
        optimizer=optimizer,
        epoch_completed=epoch_completed,
        **kwargs,
    )

    destination = (
        CHECKPOINT_DIRS[stage_name] / "latest.pt"
    )

    atomic_torch_save(
        checkpoint,
        destination,
    )

    return destination


# ==================================================
# 4. Build a separate best-model artifact
# ==================================================

def build_best_model_checkpoint(
    stage_name,
    model,
    epoch,
    metric_name,
    metric_value,
    validation_metrics=None,
):
    """
    Create an evaluation checkpoint containing
    the best-performing model state.

    This artifact is NOT used to resume training.
    """

    if stage_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown stage: {stage_name}")

    if epoch < 1 or epoch > NUM_EPOCHS:
        raise ValueError("Invalid best-model epoch.")

    if not metric_name:
        raise ValueError("A selection metric name is required.")

    if not np.isfinite(metric_value):
        raise ValueError("Selection metric must be finite.")

    checkpoint = {

        "checkpoint_version": CHECKPOINT_VERSION,

        "checkpoint_type": "best_model",

        "stage_name": stage_name,

        "model_state_dict": get_cpu_model_state(model),

        "best_epoch": int(epoch),

        "selection_metric": {
            "name": metric_name,
            "value": float(metric_value),
        },

        "validation_metrics": copy.deepcopy(
            validation_metrics if validation_metrics is not None else {}
        ),

        "experiment_config": copy.deepcopy(
            EXPERIMENT_CONFIG
        ),

        "stage_compatibility_config": copy.deepcopy(
            STAGE_COMPATIBILITY_CONFIGS[stage_name]
        ),

        "class_to_idx": copy.deepcopy(CLASS_TO_IDX),
    }

    return checkpoint


# ==================================================
# 5. Save best model
# ==================================================

def save_best_model(
    stage_name,
    model,
    epoch,
    metric_name,
    metric_value,
    validation_metrics=None,
):
    """
    Save the best model separately from latest.pt.

    The training loop will decide whether a new
    validation result is actually better.
    """

    checkpoint = build_best_model_checkpoint(
        stage_name=stage_name,
        model=model,
        epoch=epoch,
        metric_name=metric_name,
        metric_value=metric_value,
        validation_metrics=validation_metrics,
    )

    destination = (
        CHECKPOINT_DIRS[stage_name] / "best_model.pt"
    )

    atomic_torch_save(
        checkpoint,
        destination,
    )

    return destination


In [ ]:

# ==================================================
# Checkpoint validation and resumption
# ==================================================


# ==================================================
# 1. Validate a loaded training checkpoint
# ==================================================

def validate_latest_checkpoint(
    checkpoint,
    stage_name,
    optimizer,
    scheduler=None,
    scaler=None,
):
    """
    Validate checkpoint compatibility before restoring
    any training state.
    """

    required_keys = {
        "checkpoint_version",
        "checkpoint_type",
        "stage_name",
        "model_state_dict",
        "optimizer_state_dict",
        "scheduler_state_dict",
        "scaler_state_dict",
        "epoch_completed",
        "next_epoch",
        "total_epochs",
        "best_metric",
        "best_epoch",
        "training_history",
        "rng_state",
        "dataloader_generator_states",
        "stage_compatibility_config",
        "class_to_idx",
    }

    missing_keys = required_keys - set(checkpoint)

    if missing_keys:
        raise ValueError(
            f"Checkpoint missing required fields: {missing_keys}"
        )

    # Check checkpoint format.
    if checkpoint["checkpoint_version"] != CHECKPOINT_VERSION:
        raise ValueError("Unsupported checkpoint version.")

    if checkpoint["checkpoint_type"] != "latest_training_state":
        raise ValueError(
            "This is not a resumable training checkpoint. "
            "Best-model artifacts cannot be used for resumption."
        )

    # Check training stage.
    if checkpoint["stage_name"] != stage_name:
        raise ValueError(
            f"Stage mismatch: expected {stage_name}, "
            f"received {checkpoint['stage_name']}."
        )

    if stage_name not in STAGE_COMPATIBILITY_CONFIGS:
        raise ValueError(f"Unknown training stage: {stage_name}")

    # Check experiment configuration.
    expected_config = STAGE_COMPATIBILITY_CONFIGS[stage_name]

    if checkpoint["stage_compatibility_config"] != expected_config:
        raise ValueError(
            "Experiment configuration mismatch. "
            "Check architecture, dataset, transforms, "
            "optimiser and training settings."
        )

    # Check class mapping.
    if checkpoint["class_to_idx"] != CLASS_TO_IDX:
        raise ValueError("Checkpoint class mapping mismatch.")

    # Validate epoch information.
    epoch_completed = checkpoint["epoch_completed"]
    next_epoch = checkpoint["next_epoch"]
    total_epochs = checkpoint["total_epochs"]

    if (
        not isinstance(epoch_completed, int)
        or not isinstance(next_epoch, int)
        or not isinstance(total_epochs, int)
    ):
        raise ValueError("Epoch metadata must contain integers.")

    if total_epochs != NUM_EPOCHS:
        raise ValueError(
            f"Epoch target mismatch: checkpoint={total_epochs}, "
            f"current={NUM_EPOCHS}."
        )

    if not 1 <= epoch_completed <= total_epochs:
        raise ValueError("Invalid completed epoch.")

    if next_epoch != epoch_completed + 1:
        raise ValueError(
            "Inconsistent epoch metadata: next_epoch "
            "must equal epoch_completed + 1."
        )

    # Validate best-model metadata.
    best_metric = checkpoint["best_metric"]
    best_epoch = checkpoint["best_epoch"]

    if (best_metric is None) != (best_epoch is None):
        raise ValueError("Best metric and best epoch are inconsistent.")

    if best_metric is not None:

        if not np.isfinite(best_metric):
            raise ValueError("Best metric must be finite.")

        if not 1 <= best_epoch <= epoch_completed:
            raise ValueError("Invalid best epoch.")

    # Verify the optimizer is available.
    if optimizer is None:
        raise ValueError("An optimizer is required for resumption.")

    if not isinstance(checkpoint["optimizer_state_dict"], dict):
        raise ValueError("Invalid optimizer state.")

    # Scheduler compatibility.
    saved_scheduler = checkpoint["scheduler_state_dict"]

    if (scheduler is None) != (saved_scheduler is None):
        raise ValueError(
            "Scheduler configuration differs from the checkpoint."
        )

    # GradScaler compatibility.
    saved_scaler = checkpoint["scaler_state_dict"]

    if (scaler is None) != (saved_scaler is None):
        raise ValueError(
            "GradScaler configuration differs from the checkpoint."
        )

    # History validation.
    if not isinstance(checkpoint["training_history"], list):
        raise ValueError("Training history must be a list.")

    # RNG validation.
    if not isinstance(checkpoint["rng_state"], dict):
        raise ValueError("Missing or invalid RNG state.")

    # DataLoader generator validation.
    saved_generators = checkpoint["dataloader_generator_states"]

    if set(saved_generators) != set(DATA_LOADER_GENERATORS):
        raise ValueError("DataLoader generator configuration mismatch.")

    print(f"Checkpoint validation passed: {stage_name}")

    return True


# ==================================================
# 2. Load and resume a training checkpoint
# ==================================================

def load_latest_checkpoint(
    stage_name,
    model,
    optimizer,
    scheduler=None,
    scaler=None,
    checkpoint_path=None,
):
    """
    Restore the most recently completed training epoch.

    Returns metadata required by the training loop.
    """

    if checkpoint_path is None:

        if stage_name not in CHECKPOINT_DIRS:
            raise ValueError(f"Unknown training stage: {stage_name}")

        checkpoint_path = (
            CHECKPOINT_DIRS[stage_name] / "latest.pt"
        )

    checkpoint_path = Path(checkpoint_path)

    if not checkpoint_path.is_file():
        raise FileNotFoundError(
            f"No latest checkpoint found at {checkpoint_path}"
        )

    # Load onto CPU first. Model and optimizer loading
    # will place tensors appropriately for their parameters.
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=True,
    )

    # Validate EVERYTHING before changing the current state.
    validate_latest_checkpoint(
        checkpoint=checkpoint,
        stage_name=stage_name,
        optimizer=optimizer,
        scheduler=scheduler,
        scaler=scaler,
    )

    # Restore model parameters and buffers.
    model.load_state_dict(
        checkpoint["model_state_dict"],
        strict=True,
    )

    # Restore optimiser.
    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    # Restore scheduler, when used.
    if scheduler is not None:

        scheduler.load_state_dict(
            checkpoint["scheduler_state_dict"]
        )

    # Restore mixed-precision scaler, when used.
    if scaler is not None:

        scaler.load_state_dict(
            checkpoint["scaler_state_dict"]
        )

    # Restore all DataLoader generator states.
    restore_dataloader_generator_states(
        checkpoint["dataloader_generator_states"]
    )

    # Restore global RNG states LAST.
    # This protects the restored state from accidental
    # changes during the loading operations above.
    restore_rng_state(
        checkpoint["rng_state"]
    )

    # Return training progress.
    resume_info = {

        "epoch_completed": checkpoint["epoch_completed"],

        "next_epoch": checkpoint["next_epoch"],

        "total_epochs": checkpoint["total_epochs"],

        "best_metric": checkpoint["best_metric"],

        "best_epoch": checkpoint["best_epoch"],

        "history": copy.deepcopy(
            checkpoint["training_history"]
        ),

        "epoch_metrics": copy.deepcopy(
            checkpoint["epoch_metrics"]
        ),

        "is_complete": (
            checkpoint["next_epoch"] > NUM_EPOCHS
        ),
    }

    return resume_info


print("Checkpoint validation and resumption utilities defined.")


In [ ]:

from contextlib import nullcontext


# ==================================================
# 1. Precision configuration
# ==================================================

USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.float16


def autocast_context():
    """Use FP16 autocast on CUDA, otherwise ordinary FP32."""

    if USE_AMP:
        return torch.amp.autocast(
            device_type="cuda",
            dtype=AMP_DTYPE,
        )

    return nullcontext()


def create_grad_scaler():
    """Create a fresh GradScaler for each training stage."""

    return torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP,
    )


# Update the experiment configuration with precision settings.
EXPERIMENT_CONFIG["precision"] = {
    "mixed_precision": USE_AMP,
    "amp_dtype": "float16" if USE_AMP else None,
    "cpu_fallback": "float32",
}

with open(EXPERIMENT_CONFIG_PATH, "w") as file:
    json.dump(EXPERIMENT_CONFIG, file, indent=4)


# ==================================================
# 2. Create optimiser and scheduler
# ==================================================

def create_training_components(model):
    """
    Create independent optimiser, scheduler and scaler
    instances for one training stage.
    """

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=SCHEDULER_STEP_SIZE,
        gamma=SCHEDULER_GAMMA,
    )

    scaler = create_grad_scaler()

    return optimizer, scheduler, scaler


# ==================================================
# 3. Training epoch
# ==================================================

def train_one_epoch(
    stage_name,
    model,
    optimizer,
    scaler,
    teacher=None,
    max_batches=None,
):
    """
    Train one epoch for the specified experiment stage.

    max_batches is optional and is used for smoke testing.
    """

    if stage_name not in STAGE_OBJECTIVES:
        raise ValueError(f"Unknown stage: {stage_name}")

    objective = STAGE_OBJECTIVES[stage_name]

    is_distillation = objective in (
        "soft_target_kl_divergence",
        "combined_kd_and_cross_entropy",
    )

    if is_distillation and teacher is None:
        raise ValueError(
            "A teacher model is required for distillation."
        )

    model.train()

    if teacher is not None:
        teacher.eval()

        for parameter in teacher.parameters():
            parameter.requires_grad_(False)

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    total_batches = 0

    for batch_index, (images, labels) in enumerate(train_loader):

        if max_batches is not None and batch_index >= max_batches:
            break

        images = images.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        labels = labels.to(
            DEVICE,
            non_blocking=PIN_MEMORY,
        )

        optimizer.zero_grad(set_to_none=True)

        # Forward passes use AMP when enabled.
        with autocast_context():

            student_logits = model(images)

            teacher_logits = None

            if is_distillation:

                with torch.no_grad():
                    teacher_logits = teacher(images)

        # Calculate losses in FP32 for numerical stability,
        # particularly for temperature-scaled KL divergence.

        if objective == "hard_label_cross_entropy":

            loss = hard_label_loss(
                student_logits.float(),
                labels,
            )

        elif objective == "soft_target_kl_divergence":

            loss = soft_target_kd_loss(
                student_logits.float(),
                teacher_logits.float(),
            )

        elif objective == "combined_kd_and_cross_entropy":

            loss, _, _ = combined_kd_loss(
                student_logits.float(),
                teacher_logits.float(),
                labels,
            )

        if not torch.isfinite(loss):
            raise FloatingPointError(
                f"Non-finite training loss in {stage_name}."
            )

        # Backpropagation and parameter update.
        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        batch_size = labels.size(0)

        total_loss += loss.detach().item() * batch_size

        predictions = student_logits.detach().argmax(dim=1)

        total_correct += (
            predictions == labels
        ).sum().item()

        total_samples += batch_size
        total_batches += 1

    if total_samples == 0:
        raise RuntimeError("No training batches were processed.")

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "samples": total_samples,
        "batches": total_batches,
    }


# ==================================================
# 4. Validation epoch
# ==================================================

def validate_one_epoch(
    model,
    max_batches=None,
):
    """
    Evaluate a model on internal validation data.

    All models use ground-truth cross-entropy for
    a consistent validation metric.
    """

    previous_mode = model.training

    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0
    total_batches = 0

    try:

        with torch.inference_mode():

            for batch_index, (images, labels) in enumerate(
                internal_val_loader
            ):

                if max_batches is not None and batch_index >= max_batches:
                    break

                images = images.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                labels = labels.to(
                    DEVICE,
                    non_blocking=PIN_MEMORY,
                )

                with autocast_context():
                    logits = model(images)

                # Evaluation loss remains in FP32.
                loss = hard_label_loss(
                    logits.float(),
                    labels,
                )

                if not torch.isfinite(loss):
                    raise FloatingPointError(
                        "Non-finite validation loss."
                    )

                batch_size = labels.size(0)

                total_loss += loss.item() * batch_size

                predictions = logits.argmax(dim=1)

                total_correct += (
                    predictions == labels
                ).sum().item()

                total_samples += batch_size
                total_batches += 1

    finally:
        model.train(previous_mode)

    if total_samples == 0:
        raise RuntimeError("No validation batches were processed.")

    return {
        "loss": total_loss / total_samples,
        "accuracy": total_correct / total_samples,
        "samples": total_samples,
        "batches": total_batches,
    }


print("Training and validation utilities defined.")
print(f"Mixed precision enabled: {USE_AMP}")
print(f"AMP dtype: {AMP_DTYPE if USE_AMP else 'Not applicable (FP32)'}")
print("Validation criterion: ground-truth cross-entropy")
print("Best-model selection metric: validation accuracy")


In [ ]:

# ==========================================================
# 1. Finalise checkpoint compatibility and model selection
# ==========================================================

BEST_METRIC_NAME = "validation_accuracy"
BEST_METRIC_MODE = "max"

EXPERIMENT_CONFIG["model_selection"] = {
    "metric": BEST_METRIC_NAME,
    "mode": BEST_METRIC_MODE,
    "secondary_metric": "validation_loss",
}

# Precision must be part of checkpoint compatibility.
# A checkpoint trained with AMP should not silently resume
# under a different precision configuration.

for stage_config in STAGE_COMPATIBILITY_CONFIGS.values():

    stage_config["precision"] = copy.deepcopy(
        EXPERIMENT_CONFIG["precision"]
    )

    stage_config["model_selection"] = copy.deepcopy(
        EXPERIMENT_CONFIG["model_selection"]
    )


with open(EXPERIMENT_CONFIG_PATH, "w") as file:
    json.dump(EXPERIMENT_CONFIG, file, indent=4)


# ==========================================================
# 2. Preserve original initial model states
# ==========================================================

# The student initial state was captured during model setup.
# Capture the teacher's initial state before any real training.

if "TEACHER_INITIAL_STATE" not in globals():

    TEACHER_INITIAL_STATE = get_cpu_model_state(
        teacher_model
    )


def copy_state_to_cpu(state_dict):

    return {
        name: tensor.detach().cpu().clone()
        for name, tensor in state_dict.items()
    }


# Capture these only once. Re-running this section later
# must not replace the original initial states with trained weights.

if "INITIAL_MODEL_STATES" not in globals():

    student_initial_cpu = copy_state_to_cpu(
        STUDENT_INITIAL_STATE
    )

    INITIAL_MODEL_STATES = {
        "teacher": copy_state_to_cpu(
            TEACHER_INITIAL_STATE
        ),

        "student_hard": copy.deepcopy(
            student_initial_cpu
        ),

        "student_kd_soft": copy.deepcopy(
            student_initial_cpu
        ),

        "student_kd_combined": copy.deepcopy(
            student_initial_cpu
        ),
    }


assert set(INITIAL_MODEL_STATES) == set(MODEL_REGISTRY)


# ==========================================================
# 3. Reconcile best-model metadata after an interrupted save
# ==========================================================

def reconcile_best_checkpoint_metadata(
    stage_name,
    resume_info,
):
    """
    Check whether best_model.pt is newer than the latest
    checkpoint's best-metric metadata.

    This handles an interruption between saving best_model.pt
    and saving latest.pt.
    """

    best_path = CHECKPOINT_DIRS[stage_name] / "best_model.pt"

    if not best_path.is_file():
        return resume_info

    best_checkpoint = torch.load(
        best_path,
        map_location="cpu",
        weights_only=True,
    )

    if best_checkpoint["checkpoint_type"] != "best_model":
        raise ValueError("Invalid best-model checkpoint type.")

    if best_checkpoint["stage_name"] != stage_name:
        raise ValueError("Best-model stage mismatch.")

    if best_checkpoint["stage_compatibility_config"] != (
        STAGE_COMPATIBILITY_CONFIGS[stage_name]
    ):
        raise ValueError("Best-model configuration mismatch.")

    selection = best_checkpoint["selection_metric"]

    if selection["name"] != BEST_METRIC_NAME:
        raise ValueError("Best-model selection metric mismatch.")

    saved_metric = float(selection["value"])

    if not np.isfinite(saved_metric):
        raise ValueError("Best-model metric is not finite.")

    current_metric = resume_info["best_metric"]

    if (
        current_metric is None
        or saved_metric > current_metric
    ):

        resume_info["best_metric"] = saved_metric
        resume_info["best_epoch"] = best_checkpoint["best_epoch"]

    return resume_info


# ==========================================================
# 4. Prepare a fresh or resumed training stage
# ==========================================================

def prepare_training_stage(
    stage_name,
    resume=False,
    model=None,
    checkpoint_path=None,
):
    """
    Initialise the model, optimiser, scheduler and scaler.

    Fresh stage:
        Load original model weights and reset DataLoader RNG.

    Resumed stage:
        Restore the complete latest checkpoint.
    """

    if stage_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown training stage: {stage_name}")

    if model is None:
        model = MODEL_REGISTRY[stage_name]

    checkpoint_directory = CHECKPOINT_DIRS[stage_name]

    latest_path = checkpoint_directory / "latest.pt"
    best_path = checkpoint_directory / "best_model.pt"

    model.requires_grad_(True)

    if resume:

        # Optimiser and scheduler must be constructed before
        # loading their saved state dictionaries.
        optimizer, scheduler, scaler = (
            create_training_components(model)
        )

        resume_info = load_latest_checkpoint(
            stage_name=stage_name,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            checkpoint_path=checkpoint_path,
        )

        # Preserve a best model if the previous session stopped
        # between saving best_model.pt and latest.pt.
        resume_info = reconcile_best_checkpoint_metadata(
            stage_name,
            resume_info,
        )

    else:

        # Never silently overwrite an existing experiment.
        if latest_path.exists() or best_path.exists():
            raise FileExistsError(
                f"Checkpoints already exist for {stage_name}. "
                "Use resume=True to continue the existing run."
            )

        # Restore the original initial parameters and buffers.
        model.load_state_dict(
            INITIAL_MODEL_STATES[stage_name],
            strict=True,
        )

        # Reset only for a genuinely fresh stage.
        reset_dataloader_generators_for_new_stage()

        optimizer, scheduler, scaler = (
            create_training_components(model)
        )

        resume_info = {
            "epoch_completed": 0,
            "next_epoch": 1,
            "total_epochs": NUM_EPOCHS,
            "best_metric": None,
            "best_epoch": None,
            "history": [],
            "epoch_metrics": {},
            "is_complete": False,
        }

    return {
        "stage_name": stage_name,
        "model": model,
        "optimizer": optimizer,
        "scheduler": scheduler,
        "scaler": scaler,
        "resume_info": resume_info,
    }


# ==========================================================
# 5. Run and record one complete epoch
# ==========================================================

def run_training_epoch(
    stage_name,
    model,
    optimizer,
    scheduler,
    scaler,
    epoch,
    history=None,
    best_metric=None,
    best_epoch=None,
    teacher=None,
    max_train_batches=None,
    max_val_batches=None,
    save_checkpoints=True,
):
    """
    Run training, validation, scheduler update and checkpointing
    for one epoch.

    For real training, leave both max_batches arguments as None.
    """

    if stage_name not in MODEL_REGISTRY:
        raise ValueError(f"Unknown training stage: {stage_name}")

    if not 1 <= epoch <= NUM_EPOCHS:
        raise ValueError("Epoch must be within the training target.")

    if history is None:
        history = []

    # Record the LR actually used in this epoch.
    learning_rate_used = float(
        optimizer.param_groups[0]["lr"]
    )

    # ------------------------------------------------------
    # A. Training
    # ------------------------------------------------------

    train_metrics = train_one_epoch(
        stage_name=stage_name,
        model=model,
        optimizer=optimizer,
        scaler=scaler,
        teacher=teacher,
        max_batches=max_train_batches,
    )

    # ------------------------------------------------------
    # B. Full internal validation
    # ------------------------------------------------------

    validation_metrics = validate_one_epoch(
        model=model,
        max_batches=max_val_batches,
    )

    current_metric = validation_metrics["accuracy"]

    # ------------------------------------------------------
    # C. Update best-model selection
    # ------------------------------------------------------

    improved = (
        best_metric is None
        or current_metric > best_metric
    )

    if improved:
        best_metric = float(current_metric)
        best_epoch = epoch

    # ------------------------------------------------------
    # D. Record epoch history
    # ------------------------------------------------------

    epoch_metrics = {
        "epoch": epoch,

        "learning_rate": learning_rate_used,

        "train_loss": train_metrics["loss"],
        "train_accuracy": train_metrics["accuracy"],

        "validation_loss": validation_metrics["loss"],
        "validation_accuracy": validation_metrics["accuracy"],

        "training_samples": train_metrics["samples"],
        "validation_samples": validation_metrics["samples"],
    }

    history.append(epoch_metrics)

    # ------------------------------------------------------
    # E. Advance the scheduler
    # ------------------------------------------------------

    if scheduler is not None:
        scheduler.step()

    # ------------------------------------------------------
    # F. Save the best model first, when improved
    # ------------------------------------------------------

    # If an interruption happens before latest.pt is saved,
    # the resume preparation function reconciles the metric
    # from the independently saved best-model artifact.

    best_checkpoint_path = None

    if improved and save_checkpoints:

        best_checkpoint_path = save_best_model(
            stage_name=stage_name,
            model=model,
            epoch=epoch,
            metric_name=BEST_METRIC_NAME,
            metric_value=best_metric,
            validation_metrics={
                "accuracy": validation_metrics["accuracy"],
                "loss": validation_metrics["loss"],
            },
        )

    # ------------------------------------------------------
    # G. Save the latest complete training state
    # ------------------------------------------------------

    latest_checkpoint_path = None

    if save_checkpoints:

        latest_checkpoint_path = save_latest_checkpoint(
            stage_name=stage_name,
            model=model,
            optimizer=optimizer,
            scheduler=scheduler,
            scaler=scaler,
            epoch_completed=epoch,
            best_metric=best_metric,
            best_epoch=best_epoch,
            history=history,
            epoch_metrics=epoch_metrics,
        )

    return {
        "epoch": epoch,

        "train_metrics": train_metrics,

        "validation_metrics": validation_metrics,

        "epoch_metrics": epoch_metrics,

        "best_metric": best_metric,
        "best_epoch": best_epoch,

        "history": history,

        "improved": improved,

        "latest_checkpoint_path": latest_checkpoint_path,
        "best_checkpoint_path": best_checkpoint_path,
    }


print("Training-stage preparation completed.")
print("Fresh-run and resume paths defined.")
print("Epoch orchestration defined.")
print(f"Best-model criterion: {BEST_METRIC_NAME} ({BEST_METRIC_MODE})")


In [ ]:

# ==========================================================
# Teacher training stage
# ==========================================================

def run_training_stage(
    stage_name,
    resume=False,
    teacher=None,
):
    """
    Run or resume a complete training stage.

    Checkpoints are saved after every completed epoch.
    """

    stage = prepare_training_stage(
        stage_name=stage_name,
        resume=resume,
    )

    model = stage["model"]
    optimizer = stage["optimizer"]
    scheduler = stage["scheduler"]
    scaler = stage["scaler"]

    resume_info = stage["resume_info"]

    next_epoch = resume_info["next_epoch"]
    history = resume_info["history"]

    best_metric = resume_info["best_metric"]
    best_epoch = resume_info["best_epoch"]

    objective = STAGE_OBJECTIVES[stage_name]

    if (
        objective != "hard_label_cross_entropy"
        and teacher is None
    ):
        raise ValueError(
            "A trained teacher is required for KD stages."
        )

    if resume_info["is_complete"]:

        print(f"{stage_name}: training already completed.")

        return stage

    print(f"\nStarting training stage: {stage_name}")
    print(f"Objective: {objective}")
    print(f"Starting epoch: {next_epoch}")
    print(f"Total epochs: {NUM_EPOCHS}")
    print(f"Precision: {'AMP FP16' if USE_AMP else 'FP32'}")

    try:

        for epoch in range(next_epoch, NUM_EPOCHS + 1):

            result = run_training_epoch(
                stage_name=stage_name,
                model=model,
                optimizer=optimizer,
                scheduler=scheduler,
                scaler=scaler,
                epoch=epoch,
                history=history,
                best_metric=best_metric,
                best_epoch=best_epoch,
                teacher=teacher,
                save_checkpoints=True,
            )

            history = result["history"]

            best_metric = result["best_metric"]
            best_epoch = result["best_epoch"]

            train_metrics = result["train_metrics"]
            val_metrics = result["validation_metrics"]

            print(
                f"Epoch [{epoch:03d}/{NUM_EPOCHS}] | "
                f"Train Loss: {train_metrics['loss']:.4f} | "
                f"Train Acc: {train_metrics['accuracy']:.4f} | "
                f"Val Loss: {val_metrics['loss']:.4f} | "
                f"Val Acc: {val_metrics['accuracy']:.4f} | "
                f"Best Val Acc: {best_metric:.4f} "
                f"(Epoch {best_epoch}) | "
                f"LR: {optimizer.param_groups[0]['lr']:.6f}"
            )

    except KeyboardInterrupt:

        print("\nTraining interrupted.")

        print(
            "The most recently completed epoch remains "
            "saved in latest.pt."
        )

        print(
            "Resume this stage using resume=True."
        )

    print(f"\nTraining stage finished or paused: {stage_name}")

    print(f"Last completed epoch: {history[-1]['epoch'] if history else 0}")

    print(f"Best validation accuracy: {best_metric}")

    print(f"Best epoch: {best_epoch}")

    print(
        f"Latest checkpoint: "
        f"{CHECKPOINT_DIRS[stage_name] / 'latest.pt'}"
    )

    print(
        f"Best model: "
        f"{CHECKPOINT_DIRS[stage_name] / 'best_model.pt'}"
    )

    return stage


print("Training launcher defined.")


In [ ]:
teacher_training = run_training_stage(
    stage_name="teacher",
    resume=False,
)